# Просмотр запусков retrieval

Ноутбук читает локальные манифесты из `work/retrieval_runs/` и не строит индекс заново. Сначала сравнивайте варианты только при совпадении хешей корпуса и calibration-разметки. Holdout — независимая итоговая проверка; нельзя выбирать по нему лучший запуск. Если holdout различается, итоговые числа нельзя сравнивать напрямую.


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
RUNS_PATH = ROOT / 'work' / 'retrieval_runs'
manifests = sorted(RUNS_PATH.glob('*/run_manifest.json')) if RUNS_PATH.exists() else []
print(f'Сохранённых запусков: {len(manifests)}')


In [ ]:
if not manifests:
    print('Запусков нет. Сначала завершите ноутбук 02_retrieval_exploration.ipynb с корпусом и выборками.')
else:
    run_rows = []
    for manifest_path in manifests:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        selection = manifest.get('selection', {})
        input_hashes = manifest.get('input_sha256', {})
        holdout = manifest.get('holdout_metrics', {})
        run_rows.append({
            'run_id': manifest.get('run_id', manifest_path.parent.name),
            'created_at_utc': manifest.get('created_at_utc'),
            'git_revision': manifest.get('git_revision'),
            'worktree_clean': manifest.get('git_worktree_clean'),
            'method': selection.get('method'),
            'bm25_k1': selection.get('bm25', {}).get('k1'),
            'bm25_b': selection.get('bm25', {}).get('b'),
            'thresholds': json.dumps(selection.get('thresholds_by_language', {}), ensure_ascii=False, sort_keys=True),
            'corpus_sha256': input_hashes.get('corpus'),
            'calibration_sha256': input_hashes.get('calibration_queries'),
            'holdout_sha256': input_hashes.get('holdout_queries'),
            'holdout_recall_at_5': holdout.get('recall_at_5'),
            'holdout_mrr': holdout.get('mrr'),
            'holdout_query_count': holdout.get('query_count'),
        })
    runs = pd.DataFrame(run_rows).sort_values('created_at_utc', na_position='last', ascending=False)
    display(runs)


## Сопоставление calibration-вариантов

Группа задаётся хешем корпуса и calibration-разметки. Внутри неё можно сравнивать калибровочные метрики запусков и выбранные настройки. Различающиеся корпуса или calibration-файлы означают разные эксперименты. Holdout-метрики показаны для аудита, но не должны влиять на выбор метода и порога.


In [ ]:
if manifests:
    calibration_rows = []
    for manifest_path in manifests:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        input_hashes = manifest.get('input_sha256', {})
        group = f"{input_hashes.get('corpus')} | {input_hashes.get('calibration_queries')}"
        for metric in manifest.get('calibration_method_metrics', []):
            calibration_rows.append({
                'run_id': manifest.get('run_id', manifest_path.parent.name),
                'data_group': group,
                'method': metric.get('method'),
                'recall_at_5': metric.get('Recall@5'),
                'mrr': metric.get('MRR'),
                'holdout_sha256': input_hashes.get('holdout_queries'),
            })
    if calibration_rows:
        calibration_runs = pd.DataFrame(calibration_rows).sort_values(['data_group', 'mrr', 'recall_at_5'], ascending=[True, False, False])
        display(calibration_runs)
    else:
        print('В сохранённых манифестах нет calibration-метрик. Откройте исходный CSV запуска.')


## Просмотр ошибок конкретного запуска

Задайте `SELECTED_RUN_ID` из таблицы. Таблица содержит текст calibration-запросов и может включать чувствительные данные. Holdout-кейсы здесь намеренно не используются для настройки.


In [ ]:
SELECTED_RUN_ID = None
if SELECTED_RUN_ID is None:
    print('Укажите run_id для просмотра calibration ошибок.')
else:
    error_path = RUNS_PATH / SELECTED_RUN_ID / 'calibration_error_cases.csv'
    if error_path.is_file():
        display(pd.read_csv(error_path))
    else:
        print('Для этого запуска таблица ошибок отсутствует.')
